In [ ]:
import os
import tifffile as tiff
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from scipy import ndimage
from skimage.measure import label, regionprops
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
import re
import seaborn as sns
from skimage.filters import threshold_otsu, threshold_triangle, threshold_yen, threshold_li
from tqdm import tqdm

In [ ]:
#Test thresholding method.... Otsu is likely the best

In [ ]:
# Load image (assuming first slice is nuclei)
file_path = "/Volumes/Kaja/Emily/Flebogamma/20250702 FBG 2% Plate 1 4 Hours/20250702 Plate 1 Images/Read 2_DAPI 377,447+Read 2_Texas Red 586,647_A1_2_001.tif"
img_stack = tiff.imread(file_path)
nuclei_img = img_stack[0]

# Dictionary of thresholding methods to test
methods = {
    "Otsu": threshold_otsu,
    "Triangle": threshold_triangle,
    "Yen": threshold_yen,
    "Li": threshold_li,
}

# Apply thresholding and create binary masks
binaries = {}
for name, func in methods.items():
    thresh_val = func(nuclei_img)
    binaries[name] = nuclei_img > thresh_val

# Plot side by side for visual comparison
fig, axes = plt.subplots(1, len(binaries) + 1, figsize=(20, 6))

# Show raw image with good contrast
axes[0].imshow(nuclei_img, cmap='gray', vmin=np.percentile(nuclei_img, 1), vmax=np.percentile(nuclei_img, 99))
axes[0].set_title("Raw (16-bit)")
axes[0].axis('off')

# Show each binary mask
for ax, (name, binary) in zip(axes[1:], binaries.items()):
    ax.imshow(binary, cmap='gray')
    ax.set_title(name)
    ax.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
#Test image processing and nuclei counting

In [ ]:
# === Load image ===
file_path = "/Volumes/Kaja/Emily/Flebogamma/20250702 FBG 2% Plate 1 4 Hours/20250702 Plate 1 Images/Read 2_DAPI 377,447+Read 2_Texas Red 586,647_A1_2_001.tif"
img_stack = tiff.imread(file_path)
nuclei_img = img_stack[0]  # Assuming first slice is nuclei (16-bit)

# === Threshold (Otsu) ===
thresh_val = threshold_otsu(nuclei_img)
binary = nuclei_img > thresh_val

# === Fill holes ===
filled = ndimage.binary_fill_holes(binary)
#filled = binary

# === Distance transform ===
distance = ndimage.distance_transform_edt(filled)

# === Find local maxima for watershed seeds ===
coords = peak_local_max(distance, labels=filled, footprint=np.ones((40, 40)))
local_maxi = np.zeros_like(distance, dtype=bool)
local_maxi[tuple(coords.T)] = True
markers = label(local_maxi)

# === Apply skimage watershed ===
labels_ws = watershed(-distance, markers, mask=filled)

# === Size filtering (area in pixels) ===
# Set your min and max allowed object size (in pixels)
min_area = 150    # e.g. exclude tiny specks
max_area = 20000   # e.g. exclude huge blobs / merged objects

regions = regionprops(labels_ws)

# Get labels of regions within the desired size range
valid_labels = [r.label for r in regions if min_area <= r.area <= max_area]

# Create a filtered mask containing only valid objects
filtered_mask = np.isin(labels_ws, valid_labels)

# === Create binary processed mask from filtered objects ===
processed_binary = (filtered_mask.astype(np.uint8) * 255)

# === Overlay with label numbers ===
overlay = cv2.cvtColor(processed_binary, cv2.COLOR_GRAY2BGR)

# Use only filtered regions for numbering
filtered_regions = [r for r in regions if r.label in valid_labels]

for i, region in enumerate(filtered_regions):
    y, x = region.centroid
    cv2.putText(overlay, str(i+1), (int(x), int(y)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

# === Plot panels ===
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Normalize for display only
axes[0].imshow(nuclei_img, cmap='gray',
               vmin=np.percentile(nuclei_img, 1),
               vmax=np.percentile(nuclei_img, 99))
axes[0].set_title("Raw (16-bit)")

axes[1].imshow(filled, cmap='gray')
axes[1].set_title("Otsu Threshold")

axes[2].imshow(processed_binary, cmap='gray')
axes[2].set_title("Processed (size-filtered)")

axes[3].imshow(overlay)
axes[3].set_title(f"Nuclei Count: {len(filtered_regions)}")

for ax in axes:
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
#Loop through all folder

In [ ]:
 # === Global well-extraction helper === 
def extract_well(filename): 
    """ 
    Extract well IDs like A1, H12 from filenames containing '_A1_' etc. 
    Returns None if no match. 
    """ 
    match = re.search(r'_([A-H][0-9]{1,2})_', filename) 
    return match.group(1) if match else None 
    
# === Parameters === 
parent_directory = "/Volumes/Kaja/Emily/Flebogamma/"  # folder that contains (possibly nested) subfolders 
show_image = False 
save_image = False  # controls whether validation PNGs are written 

# === Size filtering parameters (area in pixels) === 
min_area = 150      # exclude tiny specks 
max_area = 20000    # exclude very large/merged objects 
sns.set(style="whitegrid") 

# === Loop over ALL subdirectories (any depth) === 

for subfolder_path, dirnames, filenames in os.walk(parent_directory): 
    # Skip the top-level parent itself; only process its subdirectories 
    if os.path.abspath(subfolder_path) == os.path.abspath(parent_directory): 
        continue 
    subfolder = os.path.basename(subfolder_path) 

    # Skip any analyses folders from previous runs 
    if subfolder.endswith("_analyses"): 
        print(f"Skipping analyses folder: {subfolder_path}") 
        continue 
    print(f"\n=== Processing subfolder: {subfolder_path} ===") 

    # --- Get image list (exclude AppleDouble "._" files and non-files) --- 

    image_files = [ 
        f for f in os.listdir(subfolder_path) 
        if f.lower().endswith((".tif", ".tiff")) 
        and not f.startswith("._") 
        and os.path.isfile(os.path.join(subfolder_path, f)) 
    ] 
    
    image_files.sort() 
    total_files = len(image_files) 
    print(f"Found {total_files} image files after filtering in {subfolder_path}.\n") 

    # If no TIFFs, do NOT create an analyses folder; just skip 
    if total_files == 0: 
        print(f"No TIFF images found in {subfolder_path}, skipping.") 
        continue 
        
    # Now we know this subfolder has TIFFs → create analyses dir 
    save_directory = os.path.join(subfolder_path, f"{subfolder}_analyses") 
    os.makedirs(save_directory, exist_ok=True) 
    
    # Prepare DataFrame for THIS subfolder (per-image metrics) 
    Data = pd.DataFrame(columns=[ 
        "File name", 
        "Well", 
        "Nuclei count", 
        "Texas red mean intensity", 
        "Normalized mean intensity", 
        "Texas red sum intensity", 
        "Normalized sum intensity" 
    ]) 

    skipped = [] 
    
    # === Image processing loop with robust loading === 
    for filename in tqdm(image_files, desc=f"Processing images in {subfolder}"): 
        filepath = os.path.join(subfolder_path, filename) 

        # quick signature check to avoid trying to open obvious non-TIFFs 
        try: 
            with open(filepath, "rb") as fh: 
                sig = fh.read(4) 
            if sig not in (b"II*\x00", b"MM\x00*"): 
                skipped.append((filename, "Not a TIFF signature")) 
                continue 
        except Exception as e: 
            skipped.append((filename, f"Signature read error: {e}")) 
            continue 

        # safe read 
        try: 
            img_stack = tiff.imread(filepath) 
        except Exception as e: 
            skipped.append((filename, f"tifffile read error: {e}")) 
            continue 

        if getattr(img_stack, "ndim", 0) < 3 or img_stack.shape[0] != 2: 
            skipped.append((filename, f"Unexpected stack shape: {getattr(img_stack, 'shape', None)}")) 
            continue 

        nuclei_img = img_stack[0] 
        cyto_img   = img_stack[1] 

        # Nuclei segmentation with Otsu threshold 
        thresh_val = threshold_otsu(nuclei_img) 
        binary = nuclei_img > thresh_val 
        filled = ndimage.binary_fill_holes(binary)
        distance = ndimage.distance_transform_edt(filled) 
        coords = peak_local_max(distance, labels=filled, footprint=np.ones((40, 40))) 
        local_maxi = np.zeros_like(distance, dtype=bool) 

        if coords.size: 
            local_maxi[tuple(coords.T)] = True 
        markers = label(local_maxi) 
        labels_ws = watershed(-distance, markers, mask=filled)

        # === Size filtering (area in pixels) === 
        regions = regionprops(labels_ws) 
        valid_labels = [r.label for r in regions if min_area <= r.area <= max_area] 

        # mask of only valid objects 
        filtered_mask = np.isin(labels_ws, valid_labels) 
        
        # === Create binary processed mask from filtered objects === 
        processed_binary = (filtered_mask.astype(np.uint8) * 255) 

        # recompute regions list restricted to valid objects 
        filtered_regions = [r for r in regions if r.label in valid_labels] 
        nuclei_count = len(filtered_regions) 

        # === Intensity metrics === 
        cyto_mean = float(np.mean(cyto_img)) 
        cyto_sum  = float(np.sum(cyto_img)) 
        normalized_mean_intensity = cyto_mean / nuclei_count if nuclei_count > 0 else np.nan 
        normalized_sum_intensity  = cyto_sum  / nuclei_count if nuclei_count > 0 else np.nan 

        # Add row, including Well 
        Data.loc[len(Data)] = [ 
            filename, 
            extract_well(filename), 
            nuclei_count, 
            cyto_mean, 
            normalized_mean_intensity, 
            cyto_sum, 
            normalized_sum_intensity 

        ] 

        # === Overlay & plots === 
        overlay = cv2.cvtColor(processed_binary, cv2.COLOR_GRAY2BGR) 
        for i, region in enumerate(filtered_regions): 
            y, x = region.centroid 
            cv2.putText( 
                overlay, 
                str(i + 1), 
                (int(x), int(y)), 
                cv2.FONT_HERSHEY_SIMPLEX, 
                0.5, 
                (0, 255, 0), 
                1 
            ) 
        fig, axes = plt.subplots(1, 4, figsize=(20, 5)) 
        axes[0].imshow( 
            nuclei_img, 
            cmap='gray', 
            vmin=np.percentile(nuclei_img, 1), 
            vmax=np.percentile(nuclei_img, 99) 
        ) 
        axes[0].set_title("Raw Nuclei (16-bit)") 
        axes[1].imshow(filled, cmap='gray') 
        axes[1].set_title("Threshold") 
        axes[2].imshow(processed_binary, cmap='gray') 
        axes[2].set_title("Processed Mask (size-filtered)") 
        axes[3].imshow(overlay) 
        axes[3].set_title(f"Nuclei Count: {nuclei_count}") 

        for ax in axes: 
            ax.axis('off') 
        plt.tight_layout() 
        if show_image: 
            plt.show() 
        if save_image: 
            
            # save validation PNG inside this subfolder's analyses directory 
            save_path = os.path.join( 
                save_directory, 
                filename.replace(".tif", "_validation.png").replace(".tiff", "_validation.png") 
            ) 
            fig.savefig(save_path, bbox_inches='tight') 
        plt.close(fig) 

    # === Finished image-level processing for this subfolder === 
    print(f"Finished processing images in subfolder: {subfolder_path}") 
    if skipped: 
        print(f"Skipped {len(skipped)} files in {subfolder_path}:") 
        for name, reason in skipped[:10]: 
            print(f"  - {name}: {reason}") 
        if len(skipped) > 10: 
            print("  ...") 
            
    # --- Save per-image Data (Raw_data) BEFORE nuclei-count filtering --- 
    raw_csv_path = os.path.join(save_directory, "Raw_data.csv") 
    raw_h5_path  = os.path.join(save_directory, "Raw_data.h5") 
    Data.to_csv(raw_csv_path, index=False) 
    Data.to_hdf(raw_h5_path, key="data", mode="w") 

    # === FOV pooling by Well (with nuclei-count filter) === 
    # 1) Exclude images with very high nuclei counts (>300) 
    initial_rows = len(Data) 
    Data_pool = Data[Data["Nuclei count"] <= 300].copy() 
    filtered_rows = len(Data_pool) 
    print(f"Excluded {initial_rows - filtered_rows} images with Nuclei count > 300 in {subfolder_path}") 

    # If nothing left, still write empty FOV_pooled and skip onward 
    cols_to_pool = [ 
        "Nuclei count", 
        "Texas red mean intensity", 
        "Normalized mean intensity", 
        "Texas red sum intensity", 
        "Normalized sum intensity" 
    ] 

    if Data_pool.empty or Data_pool["Well"].isnull().all(): 
        print(f"No valid wells after nuclei filter in {subfolder_path}; writing empty FOV_pooled and skipping condition-level pooling.") 
        Data_Median_FOV = pd.DataFrame(columns=["Well"] + cols_to_pool) 
        fov_csv_path = os.path.join(save_directory, "FOV_pooled.csv") 
        fov_h5_path  = os.path.join(save_directory, "FOV_pooled.h5") 
        Data_Median_FOV.to_csv(fov_csv_path, index=False) 
        Data_Median_FOV.to_hdf(fov_h5_path, key="data", mode="w") 
        try: 
            print(Data_Median_FOV.head(12)) 
        except NameError: 
            pass 
        continue 

    # 2) Check for parsing success 
    if Data_pool["Well"].isnull().any(): 
        print("Warning: Some filenames could not be parsed for well information in", subfolder_path) 

    # 3) Group by Well and take the median across FOVs 
    Data_Median_FOV = Data_pool.groupby("Well")[cols_to_pool].median().reset_index() 

    # 4) Save FOV-pooled outputs into the same analyses folder 
    fov_csv_path = os.path.join(save_directory, "FOV_pooled.csv") 
    fov_h5_path  = os.path.join(save_directory, "FOV_pooled.h5") 

    Data_Median_FOV.to_csv(fov_csv_path, index=False) 
    Data_Median_FOV.to_hdf(fov_h5_path, key="data", mode="w") 

    print(f"Grouped by well and saved FOV_pooled for subfolder '{subfolder_path}'") 

    try: 
        display(Data_Median_FOV.head(12)) 
    except NameError: 
        print(Data_Median_FOV.head(12)) 

    # === Condition-level pooling using Experiment Plan Excel === 

    # Look for "*_Experiment_Plan.xlsx" inside the subfolder (ignore temp '~$' files) 

    plan_files = [ 
        f for f in os.listdir(subfolder_path) 
        if f.endswith("_Experiment_Plan.xlsx") 
        and not f.startswith("~$") 
        and os.path.isfile(os.path.join(subfolder_path, f)) 
    ] 

    if not plan_files: 
        print(f"No '*_Experiment_Plan.xlsx' file found in {subfolder_path}; skipping condition-level pooling.") 
        continue 

    if len(plan_files) > 1: 
        print(f"Warning: Multiple '*_Experiment_Plan.xlsx' files found in {subfolder_path}, using the first one: {plan_files[0]}") 

    plan_path = os.path.join(subfolder_path, plan_files[0]) 

    try: 
        plan_df = pd.read_excel(plan_path, sheet_name="Table_view") 
    except Exception as e: 
        print(f"Error reading experiment plan '{plan_files[0]}' in {subfolder_path}: {e}") 
        continue 

    # Normalize column names (strip whitespace) 
    plan_df.columns = [str(c).strip() for c in plan_df.columns] 

    required_cols = {"Well ID", "Condition"} 
    if not required_cols.issubset(set(plan_df.columns)): 
        print(f"Experiment plan in {subfolder_path} is missing required columns {required_cols}; found {plan_df.columns}. Skipping condition-level pooling.") 
        continue 

        # Build mapping Well -> Condition from the experiment plan 
    well_col = plan_df["Well ID"].astype(str).str.strip()
    cond_col = plan_df["Condition"].astype(str).str.strip()
    well_to_condition = dict(zip(well_col, cond_col))

    # --- NEW: add Condition to Raw_data (per-image) ---
    Data["Condition"] = Data["Well"].map(well_to_condition)

    # Overwrite Raw_data.csv / Raw_data.h5 so they now include Condition
    raw_csv_path = os.path.join(save_directory, "Raw_data.csv")
    raw_h5_path  = os.path.join(save_directory, "Raw_data.h5")
    Data.to_csv(raw_csv_path, index=False)
    Data.to_hdf(raw_h5_path, key="data", mode="w")

    # --- NEW: add Condition to FOV_pooled (per-well) ---
    Data_Median_FOV["Condition"] = Data_Median_FOV["Well"].map(well_to_condition)

    # Overwrite FOV_pooled.csv / FOV_pooled.h5 so they now include Condition
    fov_csv_path = os.path.join(save_directory, "FOV_pooled.csv")
    fov_h5_path  = os.path.join(save_directory, "FOV_pooled.h5")
    Data_Median_FOV.to_csv(fov_csv_path, index=False)
    Data_Median_FOV.to_hdf(fov_h5_path, key="data", mode="w")

    # Drop wells without condition assignment for condition-level summary
    Data_Median_FOV_cond = Data_Median_FOV.dropna(subset=["Condition"]).copy()

    if Data_Median_FOV_cond.empty: 
        print(f"No wells in FOV_pooled for {subfolder_path} matched the experiment plan; skipping condition-level summary.") 
        continue 

    # Group by Condition and take median across wells 
    Data_Median_Conditions = Data_Median_FOV_cond.groupby("Condition")[cols_to_pool].median().reset_index() 

    # Save condition-level results 
    final_csv_path = os.path.join(save_directory, "Final_data.csv") 
    final_h5_path  = os.path.join(save_directory, "Final_data.h5") 
    Data_Median_Conditions.to_csv(final_csv_path, index=False) 
    Data_Median_Conditions.to_hdf(final_h5_path, key="data", mode="w") 

    print(f"Condition-level data saved for subfolder '{subfolder_path}' as Final_data.csv and Final_data.h5") 

    try: 
        display(Data_Median_Conditions.head(12)) 
    except NameError: 
        print(Data_Median_Conditions.head(12)) 

    # === Plotting: per-subfolder condition-level summaries === 
    try: 
        df_plot = pd.read_csv(final_csv_path) 
    except Exception as e: 
        print(f"Error re-loading Final_data.csv for plotting in {subfolder_path}: {e}") 
        continue 

    variables = [ 
        "Nuclei count", 
        "Texas red mean intensity", 
        "Normalized mean intensity", 
        "Texas red sum intensity", 
        "Normalized sum intensity" 
    ] 

    for var in variables: 
        plt.figure(figsize=(10, 6)) 
        ax = sns.barplot( 
            data=df_plot, 
            x="Condition", 
            y=var, 
        ) 
        plt.xticks(rotation=45, ha='right') 
        plt.title(f"{var} per Condition") 
        plt.tight_layout() 

        # Save inside the subfolder's analyses folder 
        filename = f"{var.replace(' ', '_')}_plot.png" 
        plot_save_path = os.path.join(save_directory, filename) 
        plt.savefig(plot_save_path, dpi=300) 
        plt.close() 
        print(f"Saved plot: {plot_save_path}") 

print("\nAll subfolders processed.") 